# Notebook 03: Model Deployment & Serving

**Persona:** ML Engineer

**Purpose:** Demonstrate production deployment patterns:
1. Feature View promotion (dev → prod) using features-as-code
2. Batch inference using Feature Store + Model Registry
3. Online Feature Tables (OFT) for low-latency serving
4. Continuous scoring pipeline

**Prerequisites:** Run Notebooks 00, 01, and 02 first

## 1. Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    sys.path.insert(0, ".")
    from feature_definitions.config import get_session, ROLES
    session = get_session(role=ROLES["admin"])

sys.path.insert(0, ".") if "." not in sys.path else None

from feature_definitions.config import get_config, ROLES
from snowflake.ml.feature_store import FeatureStore, CreationMode

dev_cfg  = get_config("DEV")
prod_cfg = get_config("PROD")

session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {dev_cfg['warehouse']}").collect()

print(f"Role: {session.get_current_role()}")
print(f"Dev FS:  {dev_cfg['database']}.{dev_cfg['fs_schema']}")
print(f"Prod FS: {prod_cfg['database']}.{prod_cfg['fs_schema']}")

## 2. Feature View Promotion (Dev → Prod)

### The Features-as-Code Pattern

Because all Feature View definitions live in the `feature_definitions/` Python package, promotion is straightforward:

1. **Dev:** FVs registered against `FEATURE_STORE_DEMO_DEV.FEATURE_STORE`
2. **Prod:** Same code, re-registered against `FEATURE_STORE_DEMO.FEATURE_STORE`

The Feature View *definitions* are identical – only the environment parameter changes. This enables CI/CD pipelines to promote features by re-running the registration code with `env="PROD"`.

In [ ]:
# Create production Feature Store (DTs use dedicated refresh warehouse)
refresh_wh = prod_cfg.get("refresh_warehouse", "FS_REFRESH_WH")
prod_fs = FeatureStore(
    session=session,
    database=prod_cfg["database"],
    name=prod_cfg["fs_schema"],
    default_warehouse=refresh_wh,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Register entities in production
from feature_definitions.entities import register_all
prod_entities = register_all(prod_fs)
print(f"Production entities: {len(prod_entities)}")

In [ ]:
# Promote key Feature Views to production
# Using the same factory functions with env="PROD"
from feature_definitions.shared_features import (
    user_profile_features, user_purchase_aggregates, user_session_engagement,
)
from feature_definitions.churn_features import user_recency_raw, user_recency_features

promoted = []
for name, factory, kwargs in [
    ("USER_PROFILE_FEATURES",     user_profile_features,    {}),
    ("USER_PURCHASE_AGGREGATES",  user_purchase_aggregates, {"version": "V03"}),
    ("USER_SESSION_ENGAGEMENT",   user_session_engagement,  {}),
    ("USER_RECENCY_RAW",          user_recency_raw,         {}),
    ("USER_RECENCY_FEATURES",     user_recency_features,    {}),
]:
    fv = factory(session, "PROD", **kwargs)
    version = kwargs.get("version", "V01")
    try:
        fv_reg = prod_fs.register_feature_view(feature_view=fv, version=version, block=True)
        promoted.append(name)
        print(f"  ✓ {name} {version}: {fv_reg.status}")
    except Exception as e:
        if "already exists" in str(e):
            print(f"  ○ {name} {version}: already exists (skipped)")
            promoted.append(name)
        else:
            print(f"  ✗ {name} {version}: {e}")

print(f"\nPromoted {len(promoted)} Feature Views to production")

In [ ]:
# Verify production Feature Store
prod_fvs = prod_fs.list_feature_views().to_pandas()
print(f"Production Feature Views: {len(prod_fvs)}")
for _, row in prod_fvs.iterrows():
    print(f"  {row['NAME']:40s} {row['VERSION']}")

## 3. Batch Inference

Batch inference follows the same pattern as training data generation, but with a *current* spine (no historical timestamps needed for inference).

In [ ]:
# Switch to dev FS for inference demo
dev_fs = FeatureStore(
    session=session,
    database=dev_cfg["database"],
    name=dev_cfg["fs_schema"],
    default_warehouse=dev_cfg["warehouse"],
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

db = dev_cfg["database"]
src = dev_cfg["source_schema"]
inf = dev_cfg["inference_schema"]

# Create inference spine: all active users
session.sql(f"""
    CREATE OR REPLACE TABLE {db}.{inf}.CHURN_INFERENCE_SPINE AS
    SELECT USER_ID, CURRENT_TIMESTAMP() AS INFERENCE_TS
    FROM {db}.{src}.USERS
    WHERE IS_ACTIVE = TRUE
""").collect()

spine_count = session.sql(f"SELECT COUNT(*) AS N FROM {db}.{inf}.CHURN_INFERENCE_SPINE").collect()[0]["N"]
print(f"Inference spine: {spine_count} active users")

In [ ]:
# Generate inference features
spine_df = session.table(f"{db}.{inf}.CHURN_INFERENCE_SPINE")

fv_profile  = dev_fs.get_feature_view("USER_PROFILE_FEATURES", "V01")
fv_purchase = dev_fs.get_feature_view("USER_PURCHASE_AGGREGATES", "V03")
# For inference, use USER_RECENCY_FEATURES (View) — CURRENT_TIMESTAMP() is correct
# when scoring "right now". For batch historical scoring, use USER_RECENCY_RAW + enrichment.
fv_recency  = dev_fs.get_feature_view("USER_RECENCY_FEATURES", "V01")

inference_df = dev_fs.generate_training_set(
    spine_df=spine_df,
    features=[fv_profile, fv_purchase, fv_recency],
    spine_timestamp_col="INFERENCE_TS",
    join_method="cte",
)
print(f"Inference feature set: {inference_df.count()} rows")
print(f"Columns: {inference_df.columns}")

In [ ]:
# Load model from registry and run batch predictions
from snowflake.ml.registry import Registry
import pandas as pd

registry = Registry(
    session=session,
    database_name=dev_cfg["database"],
    schema_name=dev_cfg["ml_datasets_schema"],
)

# Retrieve the churn model
mv = registry.get_model("CHURN_PREDICTION").version("V01")
print(f"Model: {mv.model_name} / {mv.version_name}")

# Get inference data and predict
inf_pd = inference_df.to_pandas()
numeric_cols = inf_pd.select_dtypes(include=["number"]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != "USER_ID"]
X_inf = inf_pd[feature_cols].fillna(0)

# Use the model's predict method
predictions = mv.run(X_inf, function_name="predict_proba")
print(f"\nPredictions shape: {predictions.shape}")

# Combine with user IDs
results = pd.DataFrame({
    "USER_ID": inf_pd["USER_ID"],
    "CHURN_PROBABILITY": predictions.iloc[:, -1] if predictions.shape[1] > 1 else predictions.iloc[:, 0],
})
results = results.sort_values("CHURN_PROBABILITY", ascending=False)
print(f"\nTop 5 churn risks:")
print(results.head(5).to_string(index=False))

## 4. Online Feature Tables (OFT)

Online Feature Tables provide low-latency feature serving for real-time inference. The Feature Store automatically materialises the latest feature values into an optimised format.

In [ ]:
from snowflake.ml.feature_store.feature_view import OnlineConfig

# Enable OFT on user profile features (View-based — most common real-time lookup)
# OFT on DT-backed FVs and View-based FVs
# Note: USER_RECENCY_RAW (DT) gets OFT — serves raw timestamps for ODT pattern.
# USER_RECENCY_FEATURES (View) does NOT get OFT (Views don't support OFTs).
for fv_name in ["USER_PROFILE_FEATURES", "SESSION_BEHAVIOR_FEATURES", "USER_RECENCY_RAW"]:
    try:
        dev_fs.update_feature_view(fv_name, "V01", online_config=OnlineConfig(enable=True))
        print(f"OFT enabled: {fv_name}")
    except Exception as e:
        print(f"OFT for {fv_name}: {e}")

import time
print("\nWaiting 30s for OFT sync...")
time.sleep(30)

# Read from Online Feature Tables
print("\nUSER_PROFILE_FEATURES (online):")
fv_profile = dev_fs.get_feature_view("USER_PROFILE_FEATURES", "V01")
dev_fs.read_feature_view(fv_profile, keys=[["usr_00000001"], ["usr_00000010"]], store_type="online").show()

print("\nSESSION_BEHAVIOR_FEATURES (online — DT-backed):")
fv_sbf = dev_fs.get_feature_view("SESSION_BEHAVIOR_FEATURES", "V01")
dev_fs.read_feature_view(fv_sbf, keys=[["sess_00000001"], ["sess_00000010"]], store_type="online").show()

# USER_RECENCY_RAW OFT serves raw timestamps — client derives DAYS_SINCE at read time
print("\nUSER_RECENCY_RAW (online — raw timestamps for ODT pattern):")
fv_urf = dev_fs.get_feature_view("USER_RECENCY_RAW", "V01")
dev_fs.read_feature_view(fv_urf, keys=[["usr_00000001"], ["usr_00000010"]], store_type="online").show()

## 5. Summary

In this notebook we:

1. **Promoted** Feature Views from dev to production using the features-as-code pattern
2. **Ran batch inference** by generating features for all active users and scoring with the Churn model
3. **Created Online Feature Tables** for low-latency real-time feature serving — including View-based (`USER_PROFILE_FEATURES`) and DT-backed (`SESSION_BEHAVIOR_FEATURES`, `USER_RECENCY_RAW`) Feature Views

The `USER_RECENCY_RAW` OFT serves raw timestamps; the client derives `DAYS_SINCE_*` at read time (the **ODT pattern**). This ensures PIT correctness for both training and serving.

Proceed to **Notebook 04: Operations & Monitoring** for pipeline health checks and lineage, then **Notebook 05: Pipeline Performance** for latency measurement.